# Lab 3：給 RAG 加一個品質關卡（30 min）

## 目標
- 建立一個簡單的 RAG 系統（Retrieve → Generate）
- 加入 **Grader 節點** 過濾不相關的文件
- 比較 Naive RAG 和加入 Grader 後的差異

## 架構
```
Naive RAG:   Query → Retrieve → Generate
Graded RAG:  Query → Retrieve → Grade → Filter → Generate
```

In [ ]:
%%capture
!pip install langgraph langchain-openai langchain-core chromadb

In [ ]:
import os
from typing import TypedDict
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage
import chromadb

# --- API Key 設定 ---
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## Step 1: 建立知識庫

用 ChromaDB 建立一個小型 HP 產品知識庫。

In [ ]:
# HP 產品文件
documents = [
    "HP EliteBook 855 G10 搭載 AMD Ryzen 7 PRO 7840U 處理器，32GB DDR5 記憶體，512GB NVMe SSD。支援 WiFi 6E 和藍牙 5.3。電池續航力約 8-10 小時。",
    "HP ProBook 450 G10 採用 Intel Core i5-1335U 處理器，16GB DDR4 記憶體，256GB SSD。配備 15.6 吋 FHD IPS 螢幕，適合中小企業辦公使用。",
    "HP Pavilion 15 是消費級筆電，搭載 Intel Core i7-1355U，16GB RAM，1TB SSD。支援觸控螢幕，適合日常使用和輕度創作。",
    "HP EliteBook 系列的 BIOS 更新方式：進入 HP Support Assistant → 檢查更新 → 選擇 BIOS 更新 → 重啟安裝。請確保電源連接，切勿中途關機。",
    "HP 筆電常見的螢幕閃爍問題排除：1. 更新顯示驅動程式 2. 調整螢幕重新整理率 3. 檢查排線連接 4. 如果問題持續，聯繫 HP 維修中心。",
    "HP 台灣客服中心電話：0800-010-055，服務時間：週一至週五 9:00-18:00。線上支援可透過 support.hp.com 提交工單。",
]

doc_ids = [f"doc_{i}" for i in range(len(documents))]

# 建立 ChromaDB 集合
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="hp_knowledge",
    metadata={"hnsw:space": "cosine"},
)

# 用 OpenAI Embeddings 嵌入文件
doc_embeddings = embeddings.embed_documents(documents)

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=doc_ids,
)

print(f"知識庫建立完成！共 {collection.count()} 筆文件")

## Step 2: Naive RAG — 先跑一次看看

直接檢索 + 生成，不做任何過濾。

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """從知識庫檢索相關文件"""
    query_embedding = embeddings.embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )
    return results["documents"][0]


def generate(query: str, docs: list[str]) -> str:
    """根據檢索到的文件生成回答"""
    context = "\n\n".join([f"[文件 {i+1}] {doc}" for i, doc in enumerate(docs)])
    prompt = f"""根據以下參考文件回答使用者問題。如果文件中沒有相關資訊，請誠實說不知道。

參考文件：
{context}

使用者問題：{query}
請用繁體中文回答。"""
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


# 測試 1：好問題（知識庫有答案）
query1 = "EliteBook 855 的記憶體有多大？"
docs1 = retrieve(query1)
print("=" * 50)
print(f"問題：{query1}")
print(f"檢索到 {len(docs1)} 筆文件：")
for i, doc in enumerate(docs1):
    print(f"  [{i+1}] {doc[:60]}...")
answer1 = generate(query1, docs1)
print(f"\nNaive RAG 回答：{answer1}")

print()

# 測試 2：刁鑽問題（知識庫可能混入不相關文件）
query2 = "HP 筆電怎麼更新 BIOS？會不會影響保固？"
docs2 = retrieve(query2)
print("=" * 50)
print(f"問題：{query2}")
print(f"檢索到 {len(docs2)} 筆文件：")
for i, doc in enumerate(docs2):
    print(f"  [{i+1}] {doc[:60]}...")
answer2 = generate(query2, docs2)
print(f"\nNaive RAG 回答：{answer2}")

## Step 3: 加入 Grader 節點

Grader 的工作：對每一份檢索到的文件，用 LLM 判斷「這份文件跟問題相關嗎？」

只保留相關的文件，再送去生成回答。

### TODO
完成 `grade_document` 函數，讓 LLM 判斷文件是否與問題相關。

In [ ]:
def grade_document(query: str, document: str) -> bool:
    """用 LLM 判斷文件是否與問題相關"""
    # =============================================
    # TODO: 完成 Grader 函數（約 5 行）
    # 提示：
    # 1. 寫一個 prompt，要求 LLM 判斷文件與問題的相關性
    # 2. 要求 LLM 只回覆 "yes" 或 "no"
    # 3. 呼叫 llm.invoke() 並取得回答
    # 4. 回傳 True（相關）或 False（不相關）
    #
    # prompt = f"""判斷以下文件是否與使用者問題相關。
    # 文件：{document}
    # 問題：{query}
    # 只回覆 yes 或 no。"""
    # response = llm.invoke([HumanMessage(content=___)])
    # return "yes" in response.content.lower()
    # =============================================
    pass


# 測試 Grader
test_query = "EliteBook 855 的記憶體有多大？"
test_doc_relevant = "HP EliteBook 855 G10 搭載 AMD Ryzen 7 PRO 7840U 處理器，32GB DDR5 記憶體"
test_doc_irrelevant = "HP 台灣客服中心電話：0800-010-055"

print(f"相關文件測試：{grade_document(test_query, test_doc_relevant)}")  # 應該是 True
print(f"不相關文件測試：{grade_document(test_query, test_doc_irrelevant)}")  # 應該是 False

## Step 4: 用 Grader 過濾後再生成

### TODO
完成過濾邏輯：對每份文件呼叫 Grader，只保留相關的。

In [ ]:
def graded_rag(query: str) -> dict:
    """帶 Grader 的 RAG Pipeline"""
    # Step 1: 檢索
    docs = retrieve(query, top_k=3)
    print(f"檢索到 {len(docs)} 筆文件")

    # Step 2: 評分過濾
    # =============================================
    # TODO: 用 grade_document 過濾文件（約 3 行）
    # 提示：
    # filtered_docs = []
    # for doc in docs:
    #     if grade_document(___, ___):
    #         filtered_docs.append(doc)
    # =============================================
    filtered_docs = []  # 替換這行

    print(f"Grader 過濾後剩 {len(filtered_docs)} 筆文件")

    # Step 3: 生成
    if filtered_docs:
        answer = generate(query, filtered_docs)
    else:
        answer = "抱歉，知識庫中沒有找到與您問題相關的資訊。"

    return {
        "query": query,
        "retrieved_docs": docs,
        "filtered_docs": filtered_docs,
        "answer": answer,
    }

In [ ]:
# 比較 Naive RAG vs Graded RAG
test_queries = [
    "EliteBook 855 的記憶體有多大？",
    "HP 筆電怎麼更新 BIOS？會不會影響保固？",
]

for query in test_queries:
    print("=" * 60)
    print(f"問題：{query}")
    print("=" * 60)

    # Naive RAG
    naive_docs = retrieve(query)
    naive_answer = generate(query, naive_docs)
    print(f"\n【Naive RAG】（使用 {len(naive_docs)} 筆文件）")
    print(f"回答：{naive_answer}")

    # Graded RAG
    print(f"\n【Graded RAG】")
    graded_result = graded_rag(query)
    print(f"回答：{graded_result['answer']}")

    print()

## 觀察：加了 Grader 之後有什麼不同？

思考以下問題：
1. Grader 過濾掉了哪些文件？這些文件真的不相關嗎？
2. 過濾後的回答品質有變好嗎？
3. Grader 會不會誤殺相關文件？如何改善？
4. 增加 Grader 會增加多少 API 呼叫成本？值得嗎？

## 延伸挑戰

加入 **Hallucination Checker**：在生成回答後，再用 LLM 檢查回答是否有「幻覺」。

提示：
```python
def check_hallucination(answer: str, docs: list[str]) -> dict:
    """檢查回答是否忠於原始文件"""
    prompt = f"""判斷以下回答是否完全基於提供的文件內容。
    如果回答包含文件中沒有的資訊，標記為 hallucination。
    
    文件：{docs}
    回答：{answer}
    
    回覆格式：
    判定：faithful 或 hallucination
    理由：..."""
    ...
```

將 Checker 加入 Pipeline：`Retrieve → Grade → Filter → Generate → Check`